# Assignment 2 — Part III: Full Fine-tuning of ImageNet-pretrained ResNet18

This notebook implements **Approach B**:

```text
Raw dress image
    ↓
ImageNet-pretrained ResNet18
    ↓
Replace the original 1000-class `fc` layer
    ↓
New 10-class dress classifier
    ↓
Backpropagation through the entire network
```

Unlike the frozen DINOv2 approach, **all ResNet18 parameters are trainable**. The pretrained convolutional filters are adjusted gently using small learning rates, while the newly created classifier head learns the dress categories from scratch.

The notebook performs:

1. data loading using the same reproducible train/validation split as Part II;
2. ImageNet-compatible preprocessing;
3. inspection and adaptation of ResNet18;
4. systematic learning-rate comparison;
5. full training with early stopping;
6. comparison of constant learning rate and `ReduceLROnPlateau`;
7. saving the best model, plots, experiment tables, and final hyperparameters.

The unseen `test_2025.csv` is **not used for model selection**. It is reserved for Part IV.

## What is trained?

The original ResNet18 contains convolutional layers that were pretrained on ImageNet and a final classifier for 1000 ImageNet categories.

We replace only the shape of the final layer:

```text
Original: 512 features → 1000 ImageNet classes
New:      512 features → 10 dress classes
```

However, during optimization we pass **all model parameters** to Adam. Therefore:

- the new final layer is trained;
- the deeper ResNet blocks are trained;
- the early convolutional layers are also trained.

This is **full fine-tuning**, not feature extraction and not classifier-head-only training.

In [ ]:
# ============================================================
# 1. IMPORTS, REPRODUCIBILITY, AND DIRECTORIES
# ============================================================

from pathlib import Path
import copy
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

PROJECT_DIR = Path.cwd()

TRAIN_CSV = PROJECT_DIR / "train_2025.csv"
TEST_CSV = PROJECT_DIR / "test_2025.csv"
TRAIN_SPLIT_CSV = PROJECT_DIR / "train_split.csv"
VAL_SPLIT_CSV = PROJECT_DIR / "validation_split.csv"
IMAGE_DIR = PROJECT_DIR / "raw"

RESULTS_DIR = PROJECT_DIR / "results_resnet18"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints_resnet18"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 2. LOAD DATA AND CREATE THE SAME SPLIT AS PART II
# ============================================================

IMAGE_COLUMN = "article_id"
CATEGORY_COLUMN = "garment_types"
LABEL_COLUMN = "label"
IMAGE_EXTENSION = ".jpg"

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Missing file: {TRAIN_CSV}")

if not TEST_CSV.exists():
    raise FileNotFoundError(f"Missing file: {TEST_CSV}")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Missing image directory: {IMAGE_DIR}")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

class_names = sorted(
    train_df[CATEGORY_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

class_to_idx = {
    class_name: class_index
    for class_index, class_name in enumerate(class_names)
}

idx_to_class = {
    class_index: class_name
    for class_name, class_index in class_to_idx.items()
}

NUM_CLASSES = len(class_names)

train_df = train_df.copy()
train_df[LABEL_COLUMN] = (
    train_df[CATEGORY_COLUMN]
    .map(class_to_idx)
    .astype("int64")
)

# Reuse the exact Part-II split when it exists.
if TRAIN_SPLIT_CSV.exists() and VAL_SPLIT_CSV.exists():
    train_split_df = pd.read_csv(TRAIN_SPLIT_CSV)
    val_split_df = pd.read_csv(VAL_SPLIT_CSV)

    print("Reusing train_split.csv and validation_split.csv.")
else:
    train_split_df, val_split_df = train_test_split(
        train_df,
        test_size=0.20,
        random_state=SEED,
        stratify=train_df[LABEL_COLUMN]
    )

    train_split_df = train_split_df.reset_index(drop=True)
    val_split_df = val_split_df.reset_index(drop=True)

    train_split_df.to_csv(TRAIN_SPLIT_CSV, index=False)
    val_split_df.to_csv(VAL_SPLIT_CSV, index=False)

    print("Created the reproducible stratified split.")

required_columns = {
    IMAGE_COLUMN,
    CATEGORY_COLUMN,
    LABEL_COLUMN
}

for name, frame in [
    ("training split", train_split_df),
    ("validation split", val_split_df)
]:
    missing = required_columns.difference(frame.columns)

    if missing:
        raise ValueError(
            f"{name} is missing columns: {missing}"
        )

print("Number of classes:", NUM_CLASSES)
print("Training samples:", len(train_split_df))
print("Validation samples:", len(val_split_df))
print("Reserved test samples:", len(test_df))

print("\nClass mapping:")
for class_name, class_index in class_to_idx.items():
    print(f"{class_index}: {class_name}")

## Image preprocessing

ResNet18 was pretrained using ImageNet normalization. Each dress image is therefore:

1. converted to RGB;
2. resized while preserving its aspect ratio;
3. padded to exactly \(224 \times 224\);
4. converted to a tensor;
5. normalized with ImageNet mean and standard deviation.

The same deterministic preprocessing is used for training and validation so that the hyperparameter comparison focuses on learning rate and scheduling.

In [ ]:
# ============================================================
# 3. ASPECT-RATIO-PRESERVING RESIZE AND TRANSFORMS
# ============================================================

class ResizeLongestSideAndPad:
    def __init__(self, target_size=224, fill=0):
        self.target_size = int(target_size)
        self.fill = fill

    def __call__(self, image):
        original_width, original_height = image.size

        if original_width <= 0 or original_height <= 0:
            raise ValueError(
                f"Invalid image dimensions: {image.size}"
            )

        scale = self.target_size / max(
            original_width,
            original_height
        )

        new_width = max(
            1,
            min(
                round(original_width * scale),
                self.target_size
            )
        )

        new_height = max(
            1,
            min(
                round(original_height * scale),
                self.target_size
            )
        )

        image = TF.resize(
            image,
            size=[new_height, new_width],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True
        )

        pad_left = (self.target_size - new_width) // 2
        pad_right = self.target_size - new_width - pad_left
        pad_top = (self.target_size - new_height) // 2
        pad_bottom = self.target_size - new_height - pad_top

        return TF.pad(
            image,
            padding=[
                pad_left,
                pad_top,
                pad_right,
                pad_bottom
            ],
            fill=self.fill
        )


TARGET_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    ResizeLongestSideAndPad(
        target_size=TARGET_SIZE,
        fill=0
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print(image_transform)

In [ ]:
# ============================================================
# 4. DATASET
# ============================================================

class ImageDressDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        image_column="article_id",
        label_column="label",
        transform=None,
        image_extension=".jpg"
    ):
        self.data = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.image_column = image_column
        self.label_column = label_column
        self.transform = transform
        self.image_extension = image_extension

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        identifier = str(row[self.image_column])

        image_path = (
            self.image_dir /
            f"{identifier}{self.image_extension}"
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        with Image.open(image_path) as image:
            image = image.convert("RGB")

            if self.transform is not None:
                image = self.transform(image)

        label = int(row[self.label_column])

        return image, label


train_dataset = ImageDressDataset(
    dataframe=train_split_df,
    image_dir=IMAGE_DIR,
    image_column=IMAGE_COLUMN,
    label_column=LABEL_COLUMN,
    transform=image_transform,
    image_extension=IMAGE_EXTENSION
)

val_dataset = ImageDressDataset(
    dataframe=val_split_df,
    image_dir=IMAGE_DIR,
    image_column=IMAGE_COLUMN,
    label_column=LABEL_COLUMN,
    transform=image_transform,
    image_extension=IMAGE_EXTENSION
)

sample_image, sample_label = train_dataset[0]

print("Sample image shape:", sample_image.shape)
print("Sample label:", sample_label)
print("Sample category:", idx_to_class[sample_label])

assert sample_image.shape == (
    3,
    TARGET_SIZE,
    TARGET_SIZE
)

In [ ]:
# ============================================================
# 5. DATALOADER FACTORY
# ============================================================

NUM_WORKERS = 4 if os.name != "nt" else 0
PIN_MEMORY = torch.cuda.is_available()


def create_image_loaders(
    batch_size=32,
    seed=42
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=(NUM_WORKERS > 0),
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=(NUM_WORKERS > 0)
    )

    return train_loader, val_loader


example_train_loader, example_val_loader = (
    create_image_loaders(batch_size=32)
)

batch_images, batch_labels = next(
    iter(example_train_loader)
)

print("Batch image shape:", batch_images.shape)
print("Batch label shape:", batch_labels.shape)
print("Training batches:", len(example_train_loader))
print("Validation batches:", len(example_val_loader))

## ResNet18 selection and classifier replacement

ResNet18 is selected because it offers a practical trade-off:

- substantially fewer parameters and lower memory use than ResNet50;
- faster hyperparameter experiments;
- sufficient representational capacity for a 10-class dress task;
- direct availability of ImageNet-pretrained weights in `torchvision`.

The original `fc` layer receives 512 features and predicts 1000 ImageNet classes. It is replaced with a new linear layer predicting the 10 dress classes.

In [ ]:
# ============================================================
# 6. BUILD A FULLY TRAINABLE RESNET18
# ============================================================

def build_resnet18(
    num_classes,
    print_architecture=False
):
    weights = ResNet18_Weights.DEFAULT

    model = resnet18(weights=weights)

    classifier_in_features = model.fc.in_features

    model.fc = nn.Linear(
        classifier_in_features,
        num_classes
    )

    # Explicitly confirm full fine-tuning.
    for parameter in model.parameters():
        parameter.requires_grad = True

    if print_architecture:
        print(model)

    return model, classifier_in_features


inspection_model, classifier_in_features = (
    build_resnet18(
        num_classes=NUM_CLASSES,
        print_architecture=True
    )
)

total_parameters = sum(
    parameter.numel()
    for parameter in inspection_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in inspection_model.parameters()
    if parameter.requires_grad
)

print("\nClassifier input features:",
      classifier_in_features)
print("New classifier:", inspection_model.fc)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print(
    "All parameters trainable:",
    all(
        parameter.requires_grad
        for parameter in inspection_model.parameters()
    )
)

assert trainable_parameters == total_parameters
assert inspection_model.fc.out_features == NUM_CLASSES

del inspection_model

In [ ]:
# ============================================================
# 7. ONE-EPOCH TRAINING AND VALIDATION FUNCTIONS
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    loss_function,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples
    }


@torch.inference_mode()
def validate_one_epoch(
    model,
    data_loader,
    loss_function,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        logits = model(images)
        loss = loss_function(logits, labels)

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples
    }

## Early stopping

Validation loss is monitored after every epoch.

- Whenever validation loss improves, the current `state_dict` is saved.
- If it does not improve for the chosen number of consecutive epochs, training stops.
- The checkpoint from the lowest validation loss is then restored.

For the short learning-rate scan, a small patience is used because each trial contains only a few epochs. For the final runs, patience is set to **4**, balancing reliable convergence with the assignment's computational constraints.

In [ ]:
# ============================================================
# 8. COMPLETE FIT FUNCTION WITH EARLY STOPPING
# ============================================================

def fit_resnet18(
    experiment_name,
    learning_rate,
    batch_size=32,
    max_epochs=15,
    patience=4,
    scheduler_name=None,
    verbose=True
):
    set_seed(SEED)

    train_loader, val_loader = create_image_loaders(
        batch_size=batch_size,
        seed=SEED
    )

    model, classifier_in_features = build_resnet18(
        num_classes=NUM_CLASSES
    )

    model = model.to(device)

    loss_function = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    scheduler = None

    if scheduler_name == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=1e-7
        )
    elif scheduler_name is not None:
        raise ValueError(
            f"Unknown scheduler: {scheduler_name}"
        )

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{experiment_name}.pt"
    )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
        "learning_rate": []
    }

    best_validation_loss = float("inf")
    best_validation_accuracy = 0.0
    best_epoch = 0
    epochs_without_improvement = 0

    start_time = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]

        train_metrics = train_one_epoch(
            model=model,
            data_loader=train_loader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device
        )

        validation_metrics = validate_one_epoch(
            model=model,
            data_loader=val_loader,
            loss_function=loss_function,
            device=device
        )

        history["train_loss"].append(
            train_metrics["loss"]
        )

        history["train_accuracy"].append(
            train_metrics["accuracy"]
        )

        history["validation_loss"].append(
            validation_metrics["loss"]
        )

        history["validation_accuracy"].append(
            validation_metrics["accuracy"]
        )

        history["learning_rate"].append(current_lr)

        improved = (
            validation_metrics["loss"]
            < best_validation_loss - 1e-6
        )

        if improved:
            best_validation_loss = (
                validation_metrics["loss"]
            )

            best_validation_accuracy = (
                validation_metrics["accuracy"]
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": "resnet18",
                    "weights": "ResNet18_Weights.DEFAULT",
                    "classifier_in_features":
                        classifier_in_features,
                    "num_classes": NUM_CLASSES,
                    "class_names": class_names,
                    "class_to_idx": class_to_idx,
                    "learning_rate": learning_rate,
                    "batch_size": batch_size,
                    "scheduler": scheduler_name,
                    "best_epoch": best_epoch,
                    "best_validation_loss":
                        best_validation_loss,
                    "best_validation_accuracy":
                        best_validation_accuracy
                },
                checkpoint_path
            )
        else:
            epochs_without_improvement += 1

        if scheduler is not None:
            scheduler.step(
                validation_metrics["loss"]
            )

        if verbose:
            print(
                f"{experiment_name} | "
                f"epoch {epoch:02d}/{max_epochs} | "
                f"lr={current_lr:.2e} | "
                f"train loss={train_metrics['loss']:.4f} | "
                f"train acc={train_metrics['accuracy']:.4f} | "
                f"val loss={validation_metrics['loss']:.4f} | "
                f"val acc={validation_metrics['accuracy']:.4f}"
            )

        if epochs_without_improvement >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}; "
                    f"best epoch was {best_epoch}."
                )
            break

    elapsed_seconds = (
        time.perf_counter() - start_time
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    result = {
        "experiment_name": experiment_name,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "scheduler": scheduler_name,
        "best_epoch": best_epoch,
        "best_validation_loss":
            best_validation_loss,
        "best_validation_accuracy":
            best_validation_accuracy,
        "epochs_completed":
            len(history["validation_loss"]),
        "elapsed_seconds": elapsed_seconds,
        "checkpoint_path": str(checkpoint_path),
        "history": history
    }

    del model
    del optimizer
    del train_loader
    del val_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Guided learning-rate scan

The assignment identifies learning rate as the primary CNN hyperparameter. We therefore hold the following fixed:

- architecture: ImageNet-pretrained ResNet18;
- optimizer: Adam;
- batch size: 32;
- all network parameters trainable;
- no scheduler during this first scan.

The four requested learning-rate candidates are each trained for a short four-epoch trial. This deliberately limits computation while identifying the most promising range.

In [ ]:
# ============================================================
# 9. SHORT LEARNING-RATE SCAN
# ============================================================

LEARNING_RATES = [
    5e-4,
    1e-4,
    5e-5,
    1e-5
]

LR_SCAN_BATCH_SIZE = 32
LR_SCAN_EPOCHS = 4
LR_SCAN_PATIENCE = 2

learning_rate_runs = []

for learning_rate in LEARNING_RATES:
    experiment_name = (
        "lr_" +
        f"{learning_rate:.0e}"
        .replace("-", "minus")
        .replace("+", "plus")
    )

    print("\n" + "=" * 88)
    print("Learning-rate trial:", learning_rate)
    print("=" * 88)

    run = fit_resnet18(
        experiment_name=experiment_name,
        learning_rate=learning_rate,
        batch_size=LR_SCAN_BATCH_SIZE,
        max_epochs=LR_SCAN_EPOCHS,
        patience=LR_SCAN_PATIENCE,
        scheduler_name=None,
        verbose=True
    )

    learning_rate_runs.append(run)


learning_rate_results = pd.DataFrame([
    {
        "learning_rate": run["learning_rate"],
        "best_epoch": run["best_epoch"],
        "best_validation_loss":
            run["best_validation_loss"],
        "best_validation_accuracy":
            run["best_validation_accuracy"],
        "epochs_completed":
            run["epochs_completed"],
        "elapsed_seconds":
            run["elapsed_seconds"]
    }
    for run in learning_rate_runs
]).sort_values(
    by=[
        "best_validation_loss",
        "best_validation_accuracy"
    ],
    ascending=[True, False]
).reset_index(drop=True)

learning_rate_results.to_csv(
    RESULTS_DIR /
    "resnet18_learning_rate_results.csv",
    index=False
)

display(learning_rate_results)

BEST_LEARNING_RATE = float(
    learning_rate_results.iloc[0][
        "learning_rate"
    ]
)

print("Selected learning rate:",
      BEST_LEARNING_RATE)

In [ ]:
# ============================================================
# 10. LEARNING-RATE COMPARISON PLOTS
# ============================================================

plt.figure(figsize=(10, 6))

for run in learning_rate_runs:
    values = run["history"]["validation_loss"]
    epochs = range(1, len(values) + 1)

    plt.plot(
        epochs,
        values,
        marker="o",
        label=f"LR={run['learning_rate']:.0e}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation cross-entropy loss")
plt.title("ResNet18 learning-rate comparison: validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "resnet18_learning_rate_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(10, 6))

for run in learning_rate_runs:
    values = run["history"][
        "validation_accuracy"
    ]
    epochs = range(1, len(values) + 1)

    plt.plot(
        epochs,
        values,
        marker="o",
        label=f"LR={run['learning_rate']:.0e}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title(
    "ResNet18 learning-rate comparison: "
    "validation accuracy"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "resnet18_learning_rate_val_accuracy.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## Final constant-LR and scheduler comparison

The best learning rate from the short scan is now evaluated in two longer runs:

1. **Constant learning rate**
2. **ReduceLROnPlateau**

Both runs start independently from the same ImageNet-pretrained ResNet18 initialization, use batch size 32, and apply early stopping with patience 4. This makes the scheduler comparison fair.

In [ ]:
# ============================================================
# 11. TRAIN BEST CONSTANT-LEARNING-RATE CONFIGURATION
# ============================================================

FINAL_BATCH_SIZE = 32
FINAL_MAX_EPOCHS = 15
FINAL_PATIENCE = 4

constant_run = fit_resnet18(
    experiment_name="final_constant_lr",
    learning_rate=BEST_LEARNING_RATE,
    batch_size=FINAL_BATCH_SIZE,
    max_epochs=FINAL_MAX_EPOCHS,
    patience=FINAL_PATIENCE,
    scheduler_name=None,
    verbose=True
)

In [ ]:
# ============================================================
# 12. TRAIN THE SCHEDULER CONFIGURATION
# ============================================================

scheduler_run = fit_resnet18(
    experiment_name="final_reduce_on_plateau",
    learning_rate=BEST_LEARNING_RATE,
    batch_size=FINAL_BATCH_SIZE,
    max_epochs=FINAL_MAX_EPOCHS,
    patience=FINAL_PATIENCE,
    scheduler_name="ReduceLROnPlateau",
    verbose=True
)

In [ ]:
# ============================================================
# 13. COMPARE CONSTANT LR AND SCHEDULER
# ============================================================

scheduler_comparison = pd.DataFrame([
    {
        "configuration": "Constant learning rate",
        "learning_rate":
            constant_run["learning_rate"],
        "scheduler": "None",
        "best_epoch":
            constant_run["best_epoch"],
        "best_validation_loss":
            constant_run[
                "best_validation_loss"
            ],
        "best_validation_accuracy":
            constant_run[
                "best_validation_accuracy"
            ],
        "epochs_completed":
            constant_run["epochs_completed"],
        "elapsed_seconds":
            constant_run["elapsed_seconds"]
    },
    {
        "configuration":
            "ReduceLROnPlateau",
        "learning_rate":
            scheduler_run["learning_rate"],
        "scheduler":
            "ReduceLROnPlateau",
        "best_epoch":
            scheduler_run["best_epoch"],
        "best_validation_loss":
            scheduler_run[
                "best_validation_loss"
            ],
        "best_validation_accuracy":
            scheduler_run[
                "best_validation_accuracy"
            ],
        "epochs_completed":
            scheduler_run["epochs_completed"],
        "elapsed_seconds":
            scheduler_run["elapsed_seconds"]
    }
]).sort_values(
    by=[
        "best_validation_loss",
        "best_validation_accuracy"
    ],
    ascending=[True, False]
).reset_index(drop=True)

scheduler_comparison.to_csv(
    RESULTS_DIR /
    "resnet18_scheduler_results.csv",
    index=False
)

display(scheduler_comparison)

In [ ]:
# ============================================================
# 14. SCHEDULER COMPARISON PLOTS
# ============================================================

plt.figure(figsize=(10, 6))

for label, run in [
    ("Constant LR", constant_run),
    ("ReduceLROnPlateau", scheduler_run)
]:
    values = run["history"]["validation_loss"]
    epochs = range(1, len(values) + 1)

    plt.plot(
        epochs,
        values,
        marker="o",
        label=label
    )

plt.xlabel("Epoch")
plt.ylabel("Validation cross-entropy loss")
plt.title(
    "ResNet18 scheduler comparison: "
    "validation loss"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "resnet18_scheduler_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(10, 6))

for label, run in [
    ("Constant LR", constant_run),
    ("ReduceLROnPlateau", scheduler_run)
]:
    values = run["history"]["validation_accuracy"]
    epochs = range(1, len(values) + 1)

    plt.plot(
        epochs,
        values,
        marker="o",
        label=label
    )

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title(
    "ResNet18 scheduler comparison: "
    "validation accuracy"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "resnet18_scheduler_val_accuracy.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(10, 5))

values = scheduler_run["history"]["learning_rate"]
epochs = range(1, len(values) + 1)

plt.step(
    epochs,
    values,
    where="post"
)

plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title(
    "Learning-rate path under ReduceLROnPlateau"
)
plt.yscale("log")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR /
    "resnet18_scheduler_learning_rate.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 15. SELECT AND SAVE THE FINAL RESNET18 CHECKPOINT
# ============================================================

candidate_runs = [
    constant_run,
    scheduler_run
]

best_final_run = min(
    candidate_runs,
    key=lambda run: (
        run["best_validation_loss"],
        -run["best_validation_accuracy"]
    )
)

FINAL_CHECKPOINT = (
    CHECKPOINT_DIR /
    "final_resnet18_classifier.pt"
)

shutil.copy2(
    best_final_run["checkpoint_path"],
    FINAL_CHECKPOINT
)

print("Selected final configuration:")
print("Learning rate:",
      best_final_run["learning_rate"])
print("Scheduler:",
      best_final_run["scheduler"])
print("Best epoch:",
      best_final_run["best_epoch"])
print(
    "Best validation loss:",
    best_final_run["best_validation_loss"]
)
print(
    "Best validation accuracy:",
    best_final_run[
        "best_validation_accuracy"
    ]
)
print("Final checkpoint:",
      FINAL_CHECKPOINT)

In [ ]:
# ============================================================
# 16. FINAL HYPERPARAMETER TABLE
# ============================================================

final_hyperparameters = pd.DataFrame({
    "component": [
        "CNN model",
        "Pretraining",
        "Fine-tuning strategy",
        "Original classifier",
        "New classifier",
        "Optimizer",
        "Learning rate",
        "Batch size",
        "Scheduler",
        "Loss function",
        "Early-stopping patience",
        "Best epoch",
        "Best validation loss",
        "Best validation accuracy"
    ],
    "selected_value": [
        "ResNet18",
        "ImageNet (ResNet18_Weights.DEFAULT)",
        "All network parameters trainable",
        "Linear(512, 1000)",
        f"Linear(512, {NUM_CLASSES})",
        "Adam",
        best_final_run["learning_rate"],
        FINAL_BATCH_SIZE,
        (
            best_final_run["scheduler"]
            if best_final_run["scheduler"]
            is not None
            else "None"
        ),
        "CrossEntropyLoss",
        FINAL_PATIENCE,
        best_final_run["best_epoch"],
        best_final_run[
            "best_validation_loss"
        ],
        best_final_run[
            "best_validation_accuracy"
        ]
    ]
})

final_hyperparameters.to_csv(
    RESULTS_DIR /
    "resnet18_final_hyperparameters.csv",
    index=False
)

display(final_hyperparameters)

## Assignment-ready interpretation

ResNet18 was selected because it offers a favourable balance between representational capacity and computational cost. Its ImageNet-pretrained weights provide useful generic visual filters, while its smaller size allows several learning-rate trials within the available GPU budget.

The original 1000-class fully connected layer was replaced with a new layer containing one output for each dress category. All parameters were left trainable, so backpropagation updated both the randomly initialized classifier and the pretrained convolutional layers. Small learning rates were used to avoid destroying the useful ImageNet representation through overly large parameter updates.

Learning-rate candidates were first compared in short, controlled trials using Adam and batch size 32. The most promising learning rate was then trained for longer with early stopping. A `ReduceLROnPlateau` scheduler was compared against a constant-learning-rate baseline. The final model was selected strictly by the lowest validation loss, with validation accuracy used as a secondary criterion. The unseen test set remains reserved for the final comparison in Part IV.

In [ ]:
# ============================================================
# 17. LIST GENERATED PART-III FILES
# ============================================================

print("Result files:")

for path in sorted(RESULTS_DIR.glob("*")):
    print(" -", path)

print("\nCheckpoint files:")

for path in sorted(CHECKPOINT_DIR.glob("*")):
    print(" -", path)

## Running this notebook on the cluster

The notebook can be executed non-interactively with `jupyter nbconvert`. A separate Slurm script should:

- request one H100 GPU;
- request four CPU cores because the DataLoader uses four workers;
- activate the existing virtual environment;
- set a suitable maximum runtime;
- save an executed copy rather than overwriting the original notebook.

The executed output file should be named:

```text
assignment2_resnet18_finetuning_cluster_executed.ipynb
```